# 2D Gaussian Score Field

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Score vectors as gradients of log-density for a 2D Gaussian.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/gaussian-score/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
from torch.distributions import MultivariateNormal

torch.manual_seed(0)

# The demo's defaults: mean (0, 0), standard deviations 1.2 and 0.8, correlation 0.2
mux, muy, sx, sy, rho = 0.0, 0.0, 1.2, 0.8, 0.2
mu = torch.tensor([mux, muy], dtype=torch.float64)
Sigma = torch.tensor([[sx ** 2, rho * sx * sy], [rho * sx * sy, sy ** 2]], dtype=torch.float64)
p = MultivariateNormal(mu, Sigma)
Prec = torch.linalg.inv(Sigma)

# ── Score field s(x) = grad log p(x) = -Sigma^{-1} (x - mu) ────────────────
grid = torch.tensor([[1.5, 0.0], [0.0, 1.5], [-1.0, 1.0], [2.0, -1.5]], dtype=torch.float64, requires_grad=True)
(s_auto,) = torch.autograd.grad(p.log_prob(grid).sum(), grid)
s_formula = -(grid.detach() - mu) @ Prec.T
print("point          autograd score          -Sigma^-1 (x - mu)")
for g, a, f in zip(grid.tolist(), s_auto.tolist(), s_formula.tolist()):
    print(f"{str(g):12s}   {str([round(v, 4) for v in a]):22s}  {[round(v, 4) for v in f]}")

# The score is not just "toward the mean": it bends with the covariance
d = grid.detach()[0] - mu
cos = torch.dot(-d, s_formula[0]) / (d.norm() * s_formula[0].norm())
print(f"\nangle between the score at (1.5, 0) and the direction to the mean: "
      f"{torch.rad2deg(torch.arccos(cos)).item():.1f} degrees")

# ── Its Jacobian is constant: -Sigma^{-1}, symmetric (a gradient field) ─────
J = torch.autograd.functional.jacobian(lambda z: torch.autograd.grad(p.log_prob(z), z, create_graph=True)[0],
                                       torch.tensor([0.3, -0.2], dtype=torch.float64))
print("Jacobian of the score =", J.tolist(), "\n-Sigma^-1             =", (-Prec).tolist())

point          autograd score          -Sigma^-1 (x - mu)
[1.5, 0.0]     [-1.0851, 0.3255]       [-1.0851, 0.3255]
[0.0, 1.5]     [0.3255, -2.4414]       [0.3255, -2.4414]
[-1.0, 1.0]    [0.9404, -1.8446]       [0.9404, -1.8446]
[2.0, -1.5]    [-1.7723, 2.8754]       [-1.7723, 2.8754]

angle between the score at (1.5, 0) and the direction to the mean: 16.7 degrees
Jacobian of the score = [[-0.7233796296296297, 0.2170138888888888], [0.2170138888888888, -1.627604166666666]] 
-Sigma^-1             = [[-0.7233796296296297, 0.21701388888888884], [0.21701388888888884, -1.6276041666666663]]
